# Lab 24: Causal ML — Double Machine Learning (Diagnostic Lab)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 115 min Core + 20 min setup and submission = 135 min

---

**Learning Objectives:**
- Diagnose a hand-written cross-fitting estimator that misses a known effect, and say what each of its errors does to the estimate
- Write the cross-fitted partialling-out estimator with its standard error, and match DoubleML's `DoubleMLPLR` on the same folds
- Choose the covariates for the 401(k) question, and explain why a column caused by the treatment, or a part of the outcome, does not belong in `X`
- Read a sensitivity analysis: how strong an unobserved confounder would have to be to explain the effect away
- Estimate household-level effects (CATEs) with a causal forest, and compare them with DML run separately within income quartiles

**Dataset:** Part A uses simulated data whose true effect is 5.0. Parts B and C use the 401(k) data of Chernozhukov and Hansen (2004): 9,915 households from the 1991 SIPP, saved once from DoubleML's `fetch_401K` so that every number below stays the same all term.

**Prerequisites:** cross-validation folds with `KFold` (Lab 15), bad controls (Lab 10), random forests (Lab 19).

**Format:** Part A shows you an analyst's estimator with three errors in it and asks what they are. Write it yourself is the repair, as a function of your own; Parts B and C use it. Part B diagnoses a choice of covariates and has one blank. Part C is run-and-read. Write your answers in the text cell under each set of questions.

**Deliverable:** your written answers (Part A to the Reflection), the blank in Part B, and the required `cross_fit_dml()` function.

**Time estimate:** 115 min Core: Part A 20, Write it yourself 40, Part B 30, Part C 20, Reflection 5. With setup (10) and the README and GitHub submission (10), 135 min. This lab has no optional section.

The minutes are for a first reading: you read every comment and get stuck once or twice. The computer needs about five more minutes in Colab, most of it in Part A, after Write it yourself, in Part B's fits and in Part C.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Write it yourself (required, after Part A):** `cross_fit_dml()`, the analyst's estimator repaired, with a standard error added: about thirteen lines in an empty cell. A check cell tells you when it is right. Parts B and C call it.
> - **Part B:** one blank, the list of covariates you keep, with a check cell.
>
> **You write, in text cells:** two answers in Part A, three in Part B, two in Part C, and the Reflection.
>
> **You run and read:** everything else: the setup, the simulated data and the analyst's estimator, DoubleML's fits and the sensitivity analysis, and the causal forest with its charts. Read each cell before you run it: the questions ask about what it prints.

## Setup

The first cell installs DoubleML 0.11 and EconML 0.17 and prints the versions. EconML is installed
with `--no-deps` (without the packages it lists): it asks for a newer pandas than Colab's, and pip
would replace Colab's pandas mid-session. EconML runs on Colab's own pandas, scikit-learn and numpy;
`sparse` and `numba`, which it needs, come in on the first line. If the cell stops with an
`ImportError`, use **Runtime → Restart session** and run it again.

The second cell downloads the data as raw bytes (`urlopen(...).read()`) and checks its fingerprint,
`hashlib.md5(raw).hexdigest()`, a 32-character string that changes if one byte changes, so that you
work with the file this lab was written for. `io.BytesIO(raw)` lets pandas read the bytes as a file.
If the download fails, the `try`/`except` reads the course repository's copy, if you have one.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries
!pip install -q "doubleml>=0.11.4,<0.12" "scikit-learn>=1.6,<1.10" "sparse>=0.15.4,<0.18" "numba>=0.60,<0.69"
!pip install -q --no-deps "econml==0.17.0"

import hashlib
import io
from pathlib import Path
from urllib.request import urlopen

import doubleml
import econml
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from doubleml import DoubleMLData, DoubleMLPLR
from econml.dml import CausalForestDML
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

print(f"doubleml {doubleml.__version__}, econml {econml.__version__}, scikit-learn {sklearn.__version__}")

In [ ]:
# GIVEN — run as-is
# Setup, continued: the 401(k) data, one row per household
DATA_URL = ("https://raw.githubusercontent.com/RyanPiao/econ5200-student-notebooks/main/"
            "5200/data/lab-ch24-401k-2026-10.csv.gz")
LOCAL_COPY = "data/lab-ch24-401k-2026-10.csv.gz"
DATA_MD5 = "94f148f093f58ee78e93026e413abeb0"         # the fingerprint of the file this lab was written for

try:
    raw = urlopen(DATA_URL, timeout=60).read()
except Exception:                    # no download: use the course repository's copy, if there is one
    if not Path(LOCAL_COPY).exists():
        raise RuntimeError("The data did not download. Check that Colab is connected (top right), "
                           "then run this cell again.") from None
    raw = Path(LOCAL_COPY).read_bytes()
if hashlib.md5(raw).hexdigest() != DATA_MD5:
    raise RuntimeError("This is not the data file the lab was written for, so its numbers would not "
                       "match. Tell your instructor.")

data = pd.read_csv(io.BytesIO(raw), compression="gzip")
print(f"{data.shape[0]:,} households, {data.shape[1]} columns: {list(data.columns)}")

**Expected output:** the versions (doubleml 0.11.4, econml 0.17.0, and Colab's scikit-learn), then
```
9,915 households, 14 columns: ['nifa', 'net_tfa', 'tw', 'age', 'inc', 'fsize', 'educ', 'db', 'marr', 'twoearn', 'e401', 'p401', 'pira', 'hown']
```

---

## Part A: Manual Cross-Fitting — DIAGNOSE (20 min)

The partially linear model from class: $Y = \theta D + g(X) + \varepsilon$ and $D = m(X) + V$.
DML estimates $\theta$ in three steps, **clean $Y$, clean $D$, regress the residuals**:

1. predict $Y$ from $X$ with a machine-learning model and keep $\tilde{Y}_i = Y_i - \hat{\ell}(X_i)$;
2. predict $D$ from $X$ the same way and keep $\tilde{D}_i = D_i - \hat{m}(X_i)$;
3. regress $\tilde{Y}$ on $\tilde{D}$ through the origin: $\hat{\theta} = \sum_i \tilde{D}_i \tilde{Y}_i \,/\, \sum_i \tilde{D}_i^2$.

**Cross-fitting** splits the rows into $K$ folds. For each fold, the two models are fitted on the
other folds and give residuals for this fold's rows only, so every row's residuals come from models
that never saw it. $D$ here is 0 or 1, and a regressor still predicts it: its prediction estimates
$P(D = 1 \mid X)$, the propensity (a classifier's `predict_proba` would be the alternative).

The next cell simulates 5,000 rows with 100 covariates. The first five move $Y$; the first three also
move the chance of treatment, so treated and untreated rows differ. The true effect is
`TRUE_ATE = 5.0`. `np.random.default_rng(42)` is a seeded generator (Lab 18), `1 / (1 + np.exp(-z))`
the logistic function (Lab 17), and `.astype(float)` stores the 0s and 1s of `D_sim` as decimals.

In [ ]:
# GIVEN — run as-is
# Simulated data: 5,000 rows, 100 covariates, a confounded 0/1 treatment and a true effect of 5.0
gen = np.random.default_rng(42)
n, p = 5000, 100
TRUE_ATE = 5.0

X_sim = gen.normal(0, 1, size=(n, p))
propensity = 1 / (1 + np.exp(-(0.5 * X_sim[:, 0] + 0.3 * X_sim[:, 1] + 0.2 * X_sim[:, 2])))
D_sim = gen.binomial(1, propensity).astype(float)
Y_sim = (TRUE_ATE * D_sim
         + 2.0 * X_sim[:, 0] + 1.5 * X_sim[:, 1] + 1.0 * X_sim[:, 2]
         + 0.5 * X_sim[:, 3] + 0.3 * X_sim[:, 4]
         + gen.normal(0, 1, n))
print(f"{n} rows, {p} covariates, share treated {D_sim.mean():.4f}")

**Expected output:** `5000 rows, 100 covariates, share treated 0.5104`

**Reading the code.** `KFold(...).split(X)` (Lab 15) gives, for each fold, the positions of the
training rows and of the held-out rows. `clone(learner)` makes a fresh, unfitted copy of a
scikit-learn model, and `.fit` returns the fitted model, so `clone(learner).fit(...)` copies and fits
in one line. The learner is a forest of 100 trees, each at most 8 splits deep; the cell takes up to
half a minute.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Cross-fitted DML on the simulated data
def broken_dml(Y, D, X, learner, n_folds=2, random_state=42):
    """The analyst's DML estimate of theta."""
    Y_tilde = np.zeros(len(Y))
    D_tilde = np.zeros(len(Y))
    folds = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    for train_idx, test_idx in folds.split(X):
        model_y = clone(learner).fit(X[train_idx], Y[train_idx])
        Y_tilde[train_idx] = Y[train_idx] - model_y.predict(X[train_idx])
        D_tilde[train_idx] = D[train_idx]
    return np.mean(D_tilde * Y_tilde)


sim_forest = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
broken_ate = broken_dml(Y_sim, D_sim, X_sim, sim_forest)
print(f"true ATE {TRUE_ATE:.2f}   the analyst's estimate {broken_ate:.4f}")

**Expected output:** `true ATE 5.00   the analyst's estimate 0.7392`

### YOUR DIAGNOSIS

The analyst's function has three errors.

1. Find all three. For each, quote the line, say what it computes, and say what cross-fitted DML,
   as described at the top of this Part, computes instead.
2. Say what each error does to the estimate, and which of the three would still matter if the data
   had 100 times as many rows. Reason it out from the formulas; you need not run any variants.

*Your answers here:*

---

## Write it yourself (required — 40 min)

This function is required. No AI: write it yourself.

In the empty cell below, write the analyst's estimator repaired, with a standard error added:

`cross_fit_dml(Y, D, X, learner, n_folds=2, random_state=42)`

1. `Y` and `D` are 1-D numpy arrays of n numbers (`D` may hold the integers 0 and 1), `X` a 2-D
   numpy array of n rows, and `learner` an unfitted scikit-learn regressor, used for `D` too.
2. Make the folds with `KFold(n_splits=n_folds, shuffle=True, random_state=random_state)` and loop
   over `.split(X)`, as the analyst did.
3. In each fold, fit one fresh copy of the learner (`clone(learner)`) on the training rows to
   predict `Y` from `X`, and another fresh copy to predict `D` from `X`. For the held-out rows only,
   store the residuals $\tilde{Y} = Y - \text{prediction}$ and $\tilde{D} = D - \text{prediction}$ in
   two arrays of n decimal numbers.
4. After the loop, $\hat{\theta} = \sum_i \tilde{D}_i \tilde{Y}_i \,/\, \sum_i \tilde{D}_i^2$, over all n rows.
5. The standard error, with $u_i = \tilde{Y}_i - \hat{\theta}\tilde{D}_i$:
   $$\text{se} = \frac{\sqrt{\tfrac{1}{n}\sum_i \tilde{D}_i^2 u_i^2}}{\tfrac{1}{n}\sum_i \tilde{D}_i^2 \cdot \sqrt{n}}$$
   This is the standard error of step 4's regression that allows each row its own error variance,
   the one DoubleML reports.
6. `return theta, se`. Do not round, and do not change `Y`, `D`, `X` or `learner`: work from the
   arguments only, not from the lab's names.

Start from the analyst's function. `np.sum`, `np.mean` and `np.sqrt` do the arithmetic of steps 4
and 5. You need about thirteen lines.

The check runs your function on made-up data of 300 rows with two other learners and two settings
of the folds, and compares it with DoubleML's `DoubleMLPLR` fitted on the same folds. Until you
write the function, it says "Not written yet"; then it fails with a message until your function is
right. The cell after the check runs your function on Part A's data.

In [ ]:
# WRITE IT YOURSELF — required: write cross_fit_dml(Y, D, X, learner, n_folds=2, random_state=42)

In [ ]:
# CHECK — for cross_fit_dml above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "cross_fit_dml" not in globals():
    print("Not written yet: write cross_fit_dml in the empty cell above, then run this check again.")
else:
    import inspect
    from sklearn.linear_model import LinearRegression
    from sklearn.neighbors import KNeighborsRegressor

    def names_used(code):
        """Every global or attribute name a function's code refers to, including inner functions."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def near(a, b):
        """Equal to about six significant digits."""
        return bool(np.isclose(a, b, rtol=1e-6, atol=1e-12))

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(cross_fit_dml)
    params = list(signature.parameters)
    assert params[:6] == ["Y", "D", "X", "learner", "n_folds", "random_state"], (
        f"cross_fit_dml's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "cross_fit_dml(Y, D, X, learner, n_folds=2, random_state=42).")
    defaults = {name: signature.parameters[name].default for name in ["n_folds", "random_state"]}
    shown = {name: ("none" if value is inspect.Parameter.empty else value) for name, value in defaults.items()}
    assert defaults == {"n_folds": 2, "random_state": 42}, (
        f"cross_fit_dml's defaults are {shown}. Give n_folds a default of 2 and random_state a default "
        "of 42, as the task says.")
    used = names_used(cross_fit_dml.__code__)
    for lab_name in ["Y_sim", "D_sim", "X_sim", "TRUE_ATE", "sim_forest", "broken_ate", "data", "forest"]:
        assert lab_name not in used, (
            f"cross_fit_dml uses {lab_name}, a name from the lab. Compute everything from its arguments "
            "Y, D, X, learner, n_folds and random_state.")
    assert "broken_dml" not in used, (
        "cross_fit_dml calls the analyst's broken_dml. Write the repaired estimator in your own function.")
    own_models = {"RandomForestRegressor", "LinearRegression", "LassoCV", "Lasso",
                  "GradientBoostingRegressor"} & used
    assert not own_models, (
        f"cross_fit_dml makes its own model ({', '.join(sorted(own_models))}). Fit copies of the learner "
        "it is given: clone(learner).")
    assert not {"DoubleMLPLR", "DoubleMLData", "CausalForestDML"} & used, (
        "cross_fit_dml calls DoubleML or EconML. Write the estimator yourself: Part B compares it with "
        "DoubleMLPLR.")

    # made-up data: 300 rows, 4 covariates, and a treatment of integer 0s and 1s that depends on the first
    check_gen = np.random.default_rng(2026)
    X_chk = check_gen.normal(0, 1, (300, 4))
    D_chk = check_gen.binomial(1, 1 / (1 + np.exp(-X_chk[:, 0])))
    Y_chk = 2.0 * D_chk + X_chk[:, 0] + 0.5 * X_chk[:, 1] ** 2 + check_gen.normal(0, 1, 300)

    THETA_SLIPS = {
        "insample": "takes each row's residuals from models fitted on rows that include that row. Fit on "
                    "the training rows and compute residuals for the held-out rows (test_idx) only, so "
                    "that every residual comes from models that never saw its row",
        "nofold": "fits and predicts on all rows at once, so every residual comes from models that saw "
                  "its row. Fit on each fold's training rows and predict that fold's held-out rows",
        "no_d": "uses D itself where it needs D_tilde: D is not residualized. Predict D from X with a "
                "second copy of the learner, as for Y, and subtract the prediction",
        "no_y": "uses Y itself where it needs Y_tilde: Y is not residualized. Predict Y from X and "
                "subtract the prediction",
        "mean": "is the mean of D_tilde * Y_tilde, which is not theta. Use the ratio in the task: the sum "
                "of D_tilde * Y_tilde over the sum of D_tilde squared",
        "iv": "divides by the sum of D_tilde * D. The task's theta, DoubleMLPLR's default, divides by the "
              "sum of D_tilde squared",
        "intercept": "regresses Y_tilde on D_tilde with a constant. The task's theta has no constant: "
                     "sum(D_tilde * Y_tilde) / sum(D_tilde ** 2)",
        "covvar": "comes from np.cov over np.var: np.cov divides by n - 1 and np.var by n, and both "
                  "subtract means. Use the two sums in the task",
        "noshuffle": "uses KFold without shuffle=True, so the first rows form the first fold. Use "
                     "KFold(n_splits=n_folds, shuffle=True, random_state=random_state)",
        "stratified": "uses folds other than KFold(n_splits=n_folds, shuffle=True, "
                      "random_state=random_state)'s (StratifiedKFold's, for example). Use KFold, so that "
                      "Part B can hand DoubleMLPLR the same folds",
        "one_model": "predicts Y with a model fitted to D: one copy of the learner was fitted twice, and "
                     "its second fit replaced the first. Fit a separate clone(learner) for Y and for D",
        "round_d": "turns the prediction of D into 0 or 1. Subtract the prediction itself from D, "
                   "unrounded",
        "zeros_like": "stores D's residuals in an array of integers: np.zeros_like(D) copies D's type, "
                      "and D holds the integers 0 and 1, so each residual is cut to a whole number. Use "
                      "np.zeros(len(D))",
        "mixed_rows": "pairs each held-out row's D residual with a Y residual computed for other rows. "
                      "Compute both residuals for the same rows, test_idx, and store them there",
        "d_on_y": "computes D's residual with the model fitted to Y. Subtract the prediction of the model "
                  "fitted to D",
        "iv_raw": "puts Y and D themselves into the ratio. Use the residuals in both sums: "
                  "sum(D_tilde * Y_tilde) / sum(D_tilde ** 2)",
        "swapped_folds": "fits on the held-out fold and predicts the other rows: .split(X) gives each "
                         "fold's training rows first and its held-out rows second, so write "
                         "for train_idx, test_idx in ...",
        "last_fold": "uses only the last fold's residuals. Fill one array of n residuals, fold by fold, "
                     "and compute theta once, from all n rows, after the loop",
        "fold_mean": "averages a theta from each fold. Pool the residuals of all n rows and compute one "
                     "theta, as DoubleMLPLR does",
        "flip": "has the wrong sign. Compute both residuals the same way round: observed minus predicted",
        "k_fixed": "ignores its n_folds argument: with n_folds=3 it gives the answer for 2 folds",
        "seed_fixed": "ignores its random_state argument: with random_state=7 it gives the folds of "
                      "random_state=42",
        "both_fixed": "ignores its n_folds and random_state arguments: it gives the answer for 2 folds "
                      "and random_state=42",
    }
    SE_SLIPS = {
        "no_sqrt_n": "is sqrt(n) times too large: divide by sqrt(n), and use means, not sums, inside",
        "classical": "is the classical standard error, which assumes every row has the same error "
                     "variance. Use the task's formula, which lets it differ from row to row",
        "u_ytilde": "uses Y_tilde where it needs u = Y_tilde - theta * D_tilde",
        "n_minus_1": "divides by sqrt(n - 1). The task's divides by sqrt(n)",
        "variance": "is a variance: take its square root",
        "classical_n1": "is the classical standard error, with n - 1, which assumes every row has the "
                        "same error variance. Use the task's formula, which lets it differ from row to row",
        "sum_form": "uses sums where the task's formula has means, so it comes out sqrt(n) times too "
                    "small. Use np.mean inside, as the formula does",
        "precedence": "multiplies by sqrt(n) where the formula divides by it: put the whole denominator, "
                      "the mean times sqrt(n), in parentheses",
    }
    # The reference numbers are DoubleMLPLR's on the same folds; the others come from the slips above.
    CASES = [
        ("On made-up data with LinearRegression()",
         "lr", {}, (1.937503336, 0.1500050048),
         {"insample": 1.864821508, "nofold": 1.88328873, "no_d": 0.7596516726, "no_y": 1.874071636,
          "mean": 0.4113591456, "iv": 1.981014352, "intercept": 1.9374313, "covvar": 1.943911003,
          "noshuffle": 1.945853896, "stratified": 1.908545314, "one_model": 1.896035644,
          "round_d": 1.347895358, "last_fold": 1.89051763, "fold_mean": 1.936095548,
          "flip": -1.937503336, "zeros_like": 1.46094458, "mixed_rows": -0.08229764935,
          "d_on_y": 0.1633872684, "iv_raw": 1.91615815},
         {"no_sqrt_n": 2.598162897, "classical": 0.1518061458, "u_ytilde": 0.2044074007,
          "n_minus_1": 0.1502556399, "variance": 0.02250150147, "classical_n1": 0.1520597903,
          "sum_form": 0.008660542991, "precedence": 45.00150145}),
        ("On made-up data with KNeighborsRegressor(n_neighbors=10)",
         "knn", {}, (2.00428156, 0.139391788),
         {"insample": 1.961100578, "nofold": 1.9545348, "no_d": 1.09722271, "no_y": 2.009073379,
          "mean": 0.4959928766, "iv": 2.172231576, "intercept": 2.001989133, "covvar": 2.008684748,
          "noshuffle": 2.026767451, "stratified": 2.057295538, "one_model": 2.08639019,
          "round_d": 1.228595386, "last_fold": 1.965152716, "fold_mean": 2.003905722,
          "flip": -2.00428156, "zeros_like": 0.568105688, "mixed_rows": -0.07297824475,
          "d_on_y": 0.06721522594, "iv_raw": 2.17742493},
         {"no_sqrt_n": 2.41433659, "classical": 0.1435223575, "u_ytilde": 0.1889940952,
          "n_minus_1": 0.1396246901, "variance": 0.01943007057, "classical_n1": 0.1437621611,
          "sum_form": 0.008047788635, "precedence": 41.81753641}),
        ("On made-up data with LinearRegression(), n_folds=3, random_state=7",
         "lr", {'n_folds': 3, 'random_state': 7}, (1.856323903, 0.1480470505),
         {"insample": 1.8837939, "nofold": 1.88328873, "no_d": 0.7449068384, "no_y": 1.869801095,
          "mean": 0.3886948479, "iv": 1.863381289, "covvar": 1.86253359, "noshuffle": 1.859317116,
          "stratified": 1.862410427, "one_model": 1.873588503, "round_d": 1.300908312,
          "last_fold": 2.037997784, "fold_mean": 1.857071455, "flip": -1.856323903,
          "zeros_like": 1.354996552, "d_on_y": 0.1422550562, "iv_raw": 1.876909719,
          "swapped_folds": 1.856182646, "k_fixed": 1.873927822, "seed_fixed": 1.936413651,
          "both_fixed": 1.937503336},
         {"no_sqrt_n": 2.564250134, "classical": 0.1506672965, "u_ytilde": 0.1984452503,
          "n_minus_1": 0.1482944142, "variance": 0.02191792917, "classical_n1": 0.1509190382,
          "sum_form": 0.008547500448, "precedence": 44.41411516}),
    ]

    def run(where, learner, kwargs):
        """Call cross_fit_dml on copies of the made-up data; return (theta, se), or stop with a message."""
        Y_in, D_in, X_in = Y_chk.copy(), D_chk.copy(), X_chk.copy()
        try:
            got = cross_fit_dml(Y_in, D_in, X_in, learner, **kwargs)
        except ValueError as err:
            if "broadcast" in str(err) or "shape" in str(err):
                raise AssertionError(
                    f"{where}, cross_fit_dml stopped with ValueError: {err}. A fold's residuals do not fit "
                    "the rows they are stored in: store the held-out rows' residuals at test_idx.") from None
            raise AssertionError(f"{where}, cross_fit_dml stopped with ValueError: {err}.") from None
        except Exception as err:
            raise AssertionError(f"{where}, cross_fit_dml stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        for name, before, after in [("Y", Y_chk, Y_in), ("D", D_chk, D_in), ("X", X_chk, X_in)]:
            assert np.array_equal(before, after), (
                f"{where}, cross_fit_dml changes the {name} it is given. Store residuals in new arrays "
                "and leave the arguments as they are.")
        assert not hasattr(learner, "n_features_in_"), (
            f"{where}, cross_fit_dml fits the learner it is given, so the caller's learner comes back "
            "fitted. Fit clone(learner), a fresh copy, each time.")
        assert got is not None, f"{where}, cross_fit_dml gives back nothing: use return, not print."
        if isinstance(got, (int, float, np.floating)):
            raise AssertionError(f"{where}, cross_fit_dml returns only theta. Return both: "
                                 "return theta, se.")
        if not isinstance(got, (tuple, list)) or len(got) != 2:
            raise AssertionError(f"{where}, cross_fit_dml returns a {type(got).__name__}. Return a pair: "
                                 "return theta, se.")
        try:
            theta, se = float(got[0]), float(got[1])
        except (TypeError, ValueError):
            raise AssertionError(f"{where}, cross_fit_dml returns {got!r}: theta and se should each be "
                                 "one number.") from None
        return theta, se

    def finite(where, theta, se):
        """Stop with a message if theta or se is not a number."""
        assert np.isfinite(theta) and np.isfinite(se), (
            f"{where}, cross_fit_dml returns theta = {theta} and se = {se}. A common cause: residuals "
            "stored in an array of integers, so that each is cut to a whole number. Store them in "
            "np.zeros(len(Y)) and np.zeros(len(D)).")

    def rounded(got, want):
        """True if got is want rounded to between 1 and 6 decimals, and not want itself."""
        return not near(got, want) and any(abs(got - round(want, k)) < 1e-12 for k in range(1, 7))

    for where, learner_name, kwargs, (want_theta, want_se), theta_slips, se_slips in CASES:
        learner = LinearRegression() if learner_name == "lr" else KNeighborsRegressor(n_neighbors=10)
        theta, se = run(where, learner, kwargs)
        finite(where, theta, se)
        assert not (near(theta, want_se) and near(se, want_theta)), (
            f"{where}, cross_fit_dml returns (se, theta), swapped. Return theta first: return theta, se.")
        assert not (rounded(theta, want_theta) or rounded(se, want_se)), (
            f"{where}, cross_fit_dml's results are rounded ({theta}, {se}). Do not round inside the "
            "function: the check compares about six significant digits. Round only when you print.")
        if not near(theta, want_theta):
            for slip, value in theta_slips.items():
                assert not near(theta, value), (
                    f"{where}, theta is {theta:.6f}: your function {THETA_SLIPS[slip]}.")
            raise AssertionError(
                f"{where}, theta is {theta:.6f}; it should be {want_theta:.6f}, DoubleMLPLR's on the same "
                + ("folds. It passed with LinearRegression(): does it use the learner it is given?"
                   if learner_name == "knn" else "folds. Compare each step with the task."))
        if not near(se, want_se):
            for slip, value in se_slips.items():
                assert not near(se, value), (
                    f"{where}, theta is right and se is {se:.6f}: your se {SE_SLIPS[slip]}.")
            raise AssertionError(
                f"{where}, theta is right and se is {se:.6f}; it should be {want_se:.6f}, DoubleMLPLR's. "
                "Compare it with the task's formula, step by step.")

    print("Passed: cross_fit_dml matches DoubleMLPLR on the made-up data, with two learners and two "
          "settings of the folds.")

**Expected output:**
```
Passed: cross_fit_dml matches DoubleMLPLR on the made-up data, with two learners and two settings of the folds.
```
The next cell runs your function on Part A's data with the analyst's forest and prints a 95%
interval, θ̂ ± 1.96 se. It takes about a minute.

In [ ]:
# GIVEN — run as-is, once cross_fit_dml passes its check
# Your cross_fit_dml on Part A's data, with the analyst's forest
if "cross_fit_dml" not in globals():
    print("Write cross_fit_dml (after Part A) and pass its check first, then run this cell again.")
else:
    theta_sim, se_sim = cross_fit_dml(Y_sim, D_sim, X_sim, sim_forest)
    print(f"true ATE {TRUE_ATE:.2f}   cross_fit_dml {theta_sim:.4f}   se {se_sim:.4f}   "
          f"95% interval [{theta_sim - 1.96 * se_sim:.4f}, {theta_sim + 1.96 * se_sim:.4f}]")

**Expected output:** `true ATE 5.00   cross_fit_dml 4.9715   se 0.0393   95% interval [4.8944, 5.0486]`

---

## Part B: Package-Based DML — and the Covariate Set (30 min)

The question now is the effect of 401(k) **eligibility** (`e401`) on **net financial assets**
(`net_tfa`). Firms that offer a 401(k) differ from those that do not, and so do their workers. DML
removes the bias that flexible nuisance models would bring into $\hat{\theta}$, but it conditions on
whatever you put in `X`: choosing `X` is a question of identification, and it is yours.

| column | meaning |
|---|---|
| `net_tfa` | net total financial assets — **the outcome** |
| `e401` | eligible for a 401(k) through the employer — **the treatment** |
| `p401` | *participates* in a 401(k) |
| `nifa` | non-401(k) financial assets |
| `tw` | total wealth |
| `age`, `inc`, `educ`, `fsize`, `marr`, `twoearn`, `db`, `pira`, `hown` | age, income, years of education, family size, married, two earners, a defined-benefit pension, an IRA, owns a home |

**Reading the code.** `fit_plr(x_cols)` fits DoubleML's partially linear model with the covariates
`x_cols`. `DoubleMLData` names each column's role; `DoubleMLPLR` takes a learner for the prediction
of $Y$ (`ml_l`) and one for $D$ (`ml_m`, a regressor again, for the propensity) and by default uses
Part A's ratio. `.set_sample_splitting(all_smpls=[folds])` hands it the five folds that
`KFold(n_splits=5, shuffle=True, random_state=42)` makes, the same as your `cross_fit_dml` with
`n_folds=5`. The fitted model keeps `.coef`, `.se` and `.pval`, arrays with one entry each, hence
`[0]`. The first fit uses every other column, as many analysts would.

In [ ]:
# GIVEN — run as-is
# DoubleML's partially linear model of net_tfa on e401, on our own five folds; then every other column as X
y_col, d_col = "net_tfa", "e401"
forest = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(data))


def fit_plr(x_cols):
    """DoubleMLPLR with covariates x_cols, the forest for both predictions, fitted on our folds."""
    plr = DoubleMLPLR(DoubleMLData(data, y_col=y_col, d_cols=d_col, x_cols=x_cols),
                      ml_l=clone(forest), ml_m=clone(forest), n_folds=5)
    plr.set_sample_splitting(all_smpls=[folds])
    return plr.fit()


x_naive = [c for c in data.columns if c not in [y_col, d_col]]       # every column but Y and D
plr_naive = fit_plr(x_naive)
print(f"X ({len(x_naive)} columns): {x_naive}")
print(f"ATE {plr_naive.coef[0]:,.0f}   se {plr_naive.se[0]:,.0f}   p-value {plr_naive.pval[0]:.3f}")

**Expected output:**
```
X (12 columns): ['nifa', 'tw', 'age', 'inc', 'fsize', 'educ', 'db', 'marr', 'twoearn', 'p401', 'pira', 'hown']
ATE -835   se 483   p-value 0.084
```

**Is it the software?** The next cell runs your `cross_fit_dml` on the same data, forest and folds.
`.to_numpy(float)` gives columns as a numpy array of decimals, and `np.isclose(a, b, rtol=1e-9)` is
True when a and b agree to about nine significant digits.

In [ ]:
# GIVEN — run as-is, once cross_fit_dml passes its check
# Your cross_fit_dml beside DoubleMLPLR: the same data, forest and folds
if "cross_fit_dml" not in globals():
    print("Write cross_fit_dml (after Part A) and pass its check first, then run this cell again.")
else:
    theta_naive, se_naive = cross_fit_dml(data[y_col].to_numpy(float), data[d_col].to_numpy(float),
                                          data[x_naive].to_numpy(float), forest, n_folds=5)
    print(f"{'':>14}{'theta':>14}{'se':>12}")
    print(f"{'DoubleMLPLR':>14}{plr_naive.coef[0]:>14.4f}{plr_naive.se[0]:>12.4f}")
    print(f"{'cross_fit_dml':>14}{theta_naive:>14.4f}{se_naive:>12.4f}")
    same = np.isclose(theta_naive, plr_naive.coef[0], rtol=1e-9) and np.isclose(se_naive, plr_naive.se[0], rtol=1e-9)
    print(f"same to nine digits? {same}")

**Expected output:**
```
                       theta          se
   DoubleMLPLR     -835.3796    482.7729
 cross_fit_dml     -835.3796    482.7729
same to nine digits? True
```

### DIAGNOSE: the covariate set

Published estimates of this effect are positive and in the thousands of dollars (Poterba, Venti and
Wise 1995; Chernozhukov and Hansen 2004; Chernozhukov et al. 2018). The fit above is negative and
cannot reject zero, and your own function agrees with it, so the estimator is not the problem.

The rule has two halves. **Lab 10 gave you the first: do not condition on a variable the treatment
causes.** There you generated `m = 1.5*d + noise` and `y = 1.0*d + 2.0*m + noise`; regressing `y` on
`d` alone recovered the total effect, 4.0, and adding `m` left only the direct effect, 1.0.
Controlling for a mediator removes part of the effect, not bias. **The second half: do not condition
on a part of the outcome.** If the outcome is a sum of pieces and you hold one piece fixed, the
treatment can move the outcome only through the others.

### YOUR DIAGNOSIS

1. Which columns in `x_naive` break the rule? For each, say which half it breaks, and how it is
   related to `e401` or to `net_tfa`. Use the column table above.

*Your answers here:*

**Your task.** Fill in the blank with the list of covariates you keep, as column names in quotes,
for example `["age", "inc"]`. The check after it says whether your list follows the rule; it does not
print the answer.

In [ ]:
# YOUR TASK — Part B: fill in the blank (____)
x_cols = ____                          # the covariates you keep, as a list of column names

In [ ]:
# CHECK — for x_cols above. Run it; you need not read it.
# The columns are kept as fingerprints (hashlib.md5, as in setup), so that this cell does not give the answer away.
def fingerprint(column):
    """The first 8 characters of a column name's md5 fingerprint."""
    return hashlib.md5(column.encode()).hexdigest()[:8]


CAUSED_BY_D = {"ad04b91e"}
PART_OF_Y = {"575b2bb8", "255a5cac"}
BEFORE_D = {"7d637d27", "cf9f3fde", "a2163f0c", "88eacf45", "3ee4d5c8", "f2a46e61", "d77d5e50"}
ARGUABLE = {"30ec67d4", "c4474498"}

assert isinstance(x_cols, list) and all(isinstance(c, str) for c in x_cols), (
    f"x_cols is {x_cols!r}. Make it a list of column names in quotes, for example [\"age\", \"inc\"].")
unknown = [c for c in x_cols if c not in data.columns]
assert not unknown, f"x_cols names {unknown}, which are not columns of data. Copy the names from the table."
assert len(set(x_cols)) == len(x_cols), "x_cols names a column twice. Name each once."
assert y_col not in x_cols and d_col not in x_cols, (
    "x_cols holds the outcome or the treatment itself. X holds only the other columns.")
kept = {fingerprint(c) for c in x_cols}
assert not kept & CAUSED_BY_D, (
    "x_cols keeps a column that eligibility itself causes: it breaks the first half of the rule. Find "
    "it in the column table, drop it, and run this check again.")
n_outcome = len(kept & PART_OF_Y)
assert n_outcome == 0, (
    f"x_cols keeps {n_outcome} column(s) that are a part of net_tfa or contain it: the second half of "
    "the rule. Find them in the column table, drop them, and run this check again.")
missing = [c for c in data.columns if fingerprint(c) in BEFORE_D and c not in x_cols]
assert not missing, (
    f"x_cols leaves out {missing}: characteristics fixed before a worker is eligible that predict both "
    "eligibility and saving. Leaving them out lets them confound the estimate: put them back.")
left_out = [c for c in data.columns if fingerprint(c) in ARGUABLE and c not in x_cols]
if left_out:
    print(f"Accepted, with a choice to defend: you left out {left_out}. Holding an IRA or owning a home may "
          "itself respond to eligibility, so leaving them out can be argued: say why in your answer to "
          "Question 1. The expected outputs below use all nine covariates, so your numbers will differ.")
else:
    print(f"Passed: x_cols follows both halves of the rule ({len(x_cols)} covariates).")

**Expected output:** `Passed: x_cols follows both halves of the rule (9 covariates).`

The next cell fits the model on your covariates and prints the 95% confidence interval: `.confint()`
returns a table with one row and the interval's two ends as columns, and `.iloc[0]` takes that row by
position.

In [ ]:
# GIVEN — run as-is
# DoubleMLPLR on your covariates
dml_plr = fit_plr(x_cols)
print(f"X ({len(x_cols)} columns): {x_cols}")
print(f"ATE {dml_plr.coef[0]:,.0f}   se {dml_plr.se[0]:,.0f}   p-value {dml_plr.pval[0]:.2g}")
low, high = dml_plr.confint().iloc[0]
print(f"95% CI [{low:,.0f}, {high:,.0f}]")

**Expected output** with the nine covariates the check accepts without comment:
```
X (9 columns): ['age', 'inc', 'educ', 'fsize', 'marr', 'twoearn', 'db', 'pira', 'hown']
ATE 8,639   se 1,304   p-value 3.5e-11
95% CI [6,082, 11,195]
```
Your ATE alone cannot tell you that `X` is right: other wrong lists land close to it. Check each
column against both halves of the rule.

**One column at a time.** The next cell adds each column you dropped back to your list, alone, and
refits. `[c for c in x_naive if c not in x_cols]` lists the dropped columns, and `x_cols + [c]` is
your list with one more name at the end. It takes about a minute.

In [ ]:
# GIVEN — run as-is
# Each column you dropped, added back to your covariates on its own
print(f"{'X':<20}{'ATE':>10}{'se':>10}")
print(f"{'your x_cols':<20}{dml_plr.coef[0]:>10,.0f}{dml_plr.se[0]:>10,.0f}")
for c in [c for c in x_naive if c not in x_cols]:
    plr_c = fit_plr(x_cols + [c])
    print(f"{'+ ' + c:<20}{plr_c.coef[0]:>10,.0f}{plr_c.se[0]:>10,.0f}")

**Expected output** with the nine covariates:
```
X                          ATE        se
your x_cols              8,639     1,304
+ nifa                   9,397       511
+ tw                     7,224       856
+ p401                    -764     1,735
```

### YOUR DIAGNOSIS, continued

2. Which of the columns you dropped moves the point estimate most when it is added back, and why?
   For each of the others, say what adding it changes, and why that is a problem even where the
   estimate barely moves.

*Your answers here:*

**Sensitivity analysis.** The remaining worry is a confounder you do not observe.
`sensitivity_analysis(cf_y=0.03, cf_d=0.03)` asks what one would do if it explained 3% of the
variation in `net_tfa`, and 3% of that in `e401`, left after `X`. It gives bounds on θ under that
confounder and a 95% interval around them, and two robustness values: **RV**, how much of that
leftover variation in both a confounder would have to explain to bring the estimate to zero, and
**RVa**, the same for statistical insignificance. `.sensitivity_params` holds them as shares (0.069
is 6.9%); the cell prints them as percentages. `["theta"]["lower"]` is an array, and `[0]` its only
entry.

In [ ]:
# GIVEN — run as-is
# How strong would an unobserved confounder have to be? Your covariates, then every other column
for name, plr in [("your x_cols", dml_plr), ("every other column", plr_naive)]:
    plr.sensitivity_analysis(cf_y=0.03, cf_d=0.03)
    params = plr.sensitivity_params
    print(f"{name}: theta {plr.coef[0]:,.0f}")
    print(f"   bounds [{params['theta']['lower'][0]:,.0f}, {params['theta']['upper'][0]:,.0f}]   "
          f"95% CI on the bounds [{params['ci']['lower'][0]:,.0f}, {params['ci']['upper'][0]:,.0f}]")
    print(f"   RV {100 * params['rv'][0]:.2f}%   RVa {100 * params['rva'][0]:.2f}%")

**Expected output** with the nine covariates:
```
your x_cols: theta 8,639
   bounds [4,958, 12,320]   95% CI on the bounds [2,761, 14,506]
   RV 6.90%   RVa 5.12%
every other column: theta -835
   bounds [-2,699, 1,028]   95% CI on the bounds [-3,531, 1,809]
   RV 1.36%   RVa 0.07%
```

3. With your covariates, does a confounder of the assumed strength overturn the finding? Say what
   the RV means in words. Then read the other fit's RV and RVa: does that fit's small RV tell you
   that it is fragile to an unobserved confounder, or something else?

*Your answers here:*

---

## Part C: Causal Forests (EXTEND) (20 min)

DML estimates one average effect. A **causal forest** (`CausalForestDML`, from `econml`) estimates
a Conditional Average Treatment Effect (CATE) for each household. It residualizes $Y$ and $D$ on
$X$ by cross-fitting, as your function does; grows trees that split where the effect of $D$
differs most; and uses honesty: each tree chooses its splits on one half of its sample and estimates
the effect in each leaf on the other. It uses your `x_cols`.

**Reading the code.** `model_y` and `model_t` are the two prediction models (Part B's forest), `cv=5`
the folds, and `n_estimators=500` the causal trees. EconML calls the treatment `T`. `.effect(X_401k)`
gives one CATE per household and `.effect_interval(X_401k, alpha=0.05)` the ends of a 95% interval
for each; `(cate_low > 0) | (cate_high < 0)` is True where an interval excludes zero (`|` is "or"),
and `.mean()` of True/False values is the share that is True. The fit takes under a minute; the
histogram marks the mean CATE and Part B's DML estimate.

In [ ]:
# GIVEN — run as-is
# A causal forest on your covariates: one effect of eligibility per household
Y_401k = data[y_col].to_numpy(float)
D_401k = data[d_col].to_numpy(float)
X_401k = data[x_cols].to_numpy(float)

cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    model_t=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    n_estimators=500, min_samples_leaf=20, max_depth=10, random_state=42, cv=5)
cf.fit(Y_401k, D_401k, X=X_401k)
cate = cf.effect(X_401k)
cate_low, cate_high = cf.effect_interval(X_401k, alpha=0.05)
print(f"{len(cate):,} CATEs: mean {cate.mean():,.0f}, std {cate.std():,.0f}, "
      f"min {cate.min():,.0f}, max {cate.max():,.0f}")
print(f"share of households whose 95% interval excludes zero: {((cate_low > 0) | (cate_high < 0)).mean():.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(cate, bins=60, color="lightsteelblue", edgecolor="white")
ax.axvline(cate.mean(), color="black", label=f"mean CATE {cate.mean():,.0f}")
ax.axvline(dml_plr.coef[0], color="red", linestyle="--", label=f"DML ATE (Part B) {dml_plr.coef[0]:,.0f}")
ax.set_xlabel("CATE: effect of eligibility on net financial assets ($)")
ax.set_ylabel("households")
ax.legend()
plt.show()

**Expected output** with the nine covariates:
```
9,915 CATEs: mean 7,949, std 6,659, min -10,317, max 48,190
share of households whose 95% interval excludes zero: 0.647
```
then the histogram.

**DML within income quartiles, beside the forest.** The deck shows the effect of eligibility by
income quartile. The next cell estimates it with your `cross_fit_dml` on each quartile's households
alone, beside the forest's mean CATE and the standard deviation of its CATEs for the same
households. `pd.qcut(data["inc"], q=4, labels=[...])` labels each household with its income
quartile; the box plot shows the spread of the CATEs in each quartile, and the red points are the
quartile DML estimates with 95% intervals (`ax.errorbar`). It takes about half a minute.

In [ ]:
# GIVEN — run as-is, once cross_fit_dml passes its check
# Subgroup DML by income quartile (your function) beside the forest's CATEs for the same households
quartile = pd.qcut(data["inc"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
if "cross_fit_dml" not in globals():
    print("Write cross_fit_dml (after Part A) and pass its check first, then run this cell again.")
else:
    rows = []
    for q in ["Q1", "Q2", "Q3", "Q4"]:
        in_q = (quartile == q).to_numpy()
        theta_q, se_q = cross_fit_dml(Y_401k[in_q], D_401k[in_q], X_401k[in_q], forest, n_folds=5)
        rows.append({"quartile": q, "households": in_q.sum(),
                     "income from": data["inc"][in_q].min(), "income to": data["inc"][in_q].max(),
                     "DML theta": theta_q, "DML se": se_q,
                     "forest mean CATE": cate[in_q].mean(), "forest CATE std": cate[in_q].std()})
    by_quartile = pd.DataFrame(rows).set_index("quartile")
    print(by_quartile.round(0).astype(int).to_string())

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.boxplot([cate[(quartile == q).to_numpy()] for q in by_quartile.index],
               tick_labels=list(by_quartile.index), showfliers=False)
    ax.errorbar(range(1, 5), by_quartile["DML theta"], yerr=1.96 * by_quartile["DML se"],
                fmt="o", color="red", label="DML within the quartile, 95% interval")
    ax.set_xlabel("income quartile")
    ax.set_ylabel("effect of eligibility ($)")
    ax.legend()
    plt.show()

**Expected output** with the nine covariates:
```
          households  income from  income to  DML theta  DML se  forest mean CATE  forest CATE std
quartile                                                                                          
Q1              2481        -2652      19413       4230    1085              3352             2120
Q2              2477        19416      31476       3412    1253              4188             2315
Q3              2478        31488      48582       6999    1830              7509             3557
Q4              2479        48585     242124      19228    4041             16747             6480
```
then the box plot.

### Interpretation Questions

1. Compare the mean CATE with Part B's DML estimate. Should the two be equal? Say why they can
   differ.
2. Do the quartile DML estimates and the forest agree on how the effect changes with income? Within
   a quartile, how large is the spread of the forest's CATEs next to the gaps between quartiles, and
   what would a quartile-by-quartile analysis miss? Say also why the spread of the CATEs overstates
   the spread of the true effects.

*Your answers here:*

---

## Reflection (5 min)

In 3-5 sentences: **when would you choose DML for one average effect, and when a causal forest for
household-level effects?** Consider the question (an average policy effect, or whom to target) and
what the data can support (an ATE with a sensitivity analysis, or CATEs with wide intervals).

*Your answer here:*

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR ...]` with your own words or the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Found three errors in a hand-written double machine learning
  estimator, on simulated data with a true effect of 5.0, and wrote
  my own version: [YOUR ESTIMATE] (se [YOUR SE])
* Estimated the effect of 401(k) eligibility on net financial assets
  (9,915 households): [YOUR VALUE] with every column as a control,
  [YOUR VALUE] (95% CI [YOUR CI]) after dropping [YOUR COLUMNS]
  because [YOUR REASON]; robustness value [YOUR RV]
* Causal forest and DML by income quartile: [YOUR ONE-LINE RESULT]

**Please write a README.md entry including:**
1. Project Title: Double Machine Learning and the 401(k)
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person, with no buzzwords. Use only
the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab24-causal-ml`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab24-causal-ml`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab24-causal-ml`, branch `main`,
   commit message `Lab 24 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.